### First few cells is just data preparation

My implementation is build on top of nequip and allegro python libraries to simplify data load and preprocessing.
The wigner simbols are from e3nn
Here are the links
https://github.com/mir-group/nequip
https://github.com/mir-group/allegro

In [1]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config
import os

default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cpu',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)
import numpy as np
import random
import torch
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    #print(f"Random seed set as {seed}")

os.environ['NEQUIP_NUM_TASKS'] = '4'
# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [2]:
config = Config.from_file('./config/example_ETN_opt_MEA.yaml', defaults=default_config)

config['root'] = 'results/MEA_Allegro_2'
config['seed'] = 123456 + 8
set_seed(config['seed'])
torch.manual_seed(config['seed'])
dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

In [3]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# Some hyperparameteres
#Nc = 10 # number of chennels for F features from ETN paper
#N_rank_spec = 4 # hidden rank of reduction for type radial tensor
#config['Nc'] = Nc
#config['N_rank_spec'] = N_rank_spec

# ETN parameters
#config['d'] = 4 # dimention of the tensor train
#config['N_rank_ett'] = [4, 4, 4] # ranks of tensor train



# = Build model =
final_model = model_from_config(
    config=config, initialize=True, dataset=trainer.dataset_train
)

DEBUG:root:* Initialize Output
  ...generate file name results/MEA_Allegro_2/example/log
  ...open log file results/MEA_Allegro_2/example/log
  ...generate file name results/MEA_Allegro_2/example/metrics_epoch.csv
  ...open log file results/MEA_Allegro_2/example/metrics_epoch.csv
  ...generate file name results/MEA_Allegro_2/example/metrics_initialization.csv
  ...open log file results/MEA_Allegro_2/example/metrics_initialization.csv
  ...generate file name results/MEA_Allegro_2/example/metrics_batch_train.csv
  ...open log file results/MEA_Allegro_2/example/metrics_batch_train.csv
  ...generate file name results/MEA_Allegro_2/example/metrics_batch_val.csv
  ...open log file results/MEA_Allegro_2/example/metrics_batch_val.csv
  ...generate file name results/MEA_Allegro_2/example/best_model.pth
  ...generate file name results/MEA_Allegro_2/example/last_model.pth
  ...generate file name results/MEA_Allegro_2/example/trainer.pth
  ...generate file name results/MEA_Allegro_2/example/config

...PerSpeciesScaleShift_param = dict(
...   optional_args = {'out_field': 'atomic_energy', 'scales_trainable': False, 'shifts_trainable': False, 'default_dtype': 'float32', 'num_types': 4, 'type_names': ['Nb', 'Mo', 'Ta', 'W'], 'field': 'atomic_energy', 'shifts': tensor(-11.4157), 'scales': tensor(0.8584), 'arguments_in_dataset_units': True},
...   positional_args = {'irreps_in': {'pos': 1x1oe, 'edge_index': None, 'edge_types': 1x0ee, 'node_attrs': 4x0ee, 'node_features': 4x0ee, 'edge_embedding': 8x0ee, 'edge_cutoff': 1x0ee, 'edge_attrs': 1x0ee+1x1oe+1x2ee, 'edge_features_F': 10x0ee+10x1oe+10x2ee, 'node_features_F': 10x0ee+10x1oe+10x2ee, 'node_features_ETN': 10x0ee+10x1oe+10x2ee, 'atomic_energy': 1x0ee}})
Replace string dataset_forces_rms to 0.8583642840385437
Initially outputs are globally scaled by: 0.8583642840385437, total_energy are globally shifted by None.
PerSpeciesScaleShift's arguments were in dataset units; rescaling:
  Original scales: [Nb: 0.858364, Mo: 0.858364, Ta: 0.858

In [4]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

trainer.model = final_model

# Test configuration stores as dict of parameters
data0 = AtomicData.to_AtomicDataDict(dataset[0])

In [5]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math
# forward pass
data_new = final_model(data0)



In [6]:
instructions = []
for i in range(4):
    instructions.append([tuple(el) for el in final_model.get_buffer(f'model.model.func.etn.instructions_list_{i}').tolist()])

In [7]:
from allegro import lr_orthogonal, lr_orthogonal_ind

In [8]:
cores = trainer.model.get_submodule('model.model.func.etn.cores')
instructions = []
for i in range(4):
    instructions.append([tuple(el) for el in final_model.get_buffer(f'model.model.func.etn.instructions_list_{i}').tolist()])


ranks = [1] + final_model.get_buffer(f'model.model.func.etn.N_rank_ett').tolist() + [1]


cores_new, R = lr_orthogonal(cores, ranks, instructions)


for i in range(config['d']-1):
    cores_new_2, R = lr_orthogonal_ind(cores, ranks, instructions, i)
    cores[i] = cores_new_2[i]
    cores[i+1] = cores_new_2[i+1]

In [9]:
torch.allclose(cores_new_2[i+1], cores_new[i+1])

True

In [10]:
# Check lr orthogonality
cores_new_left = cores_new[2]
cores_new_left.flatten(0, -2).T @ cores_new_left.flatten(0, -2)

tensor([[ 3.0000e+00,  2.4447e-09, -6.2981e-08, -5.5945e-08],
        [ 2.4447e-09,  3.0000e+00, -4.6566e-08,  1.1059e-08],
        [-6.2981e-08, -4.6566e-08,  3.0000e+00,  2.0955e-08],
        [-5.5945e-08,  1.1059e-08,  2.0955e-08,  3.0000e+00]],
       grad_fn=<MmBackward0>)

In [11]:
from allegro import rl_orthogonal, rl_orthogonal_ind

cores = trainer.model.get_submodule('model.model.func.etn.cores')
instructions = []
for i in range(4):
    instructions.append([tuple(el) for el in final_model.get_buffer(f'model.model.func.etn.instructions_list_{i}').tolist()])


ranks = [1] + final_model.get_buffer(f'model.model.func.etn.N_rank_ett').tolist() + [1]


cores_new, R = rl_orthogonal(cores, ranks, instructions)

for i in range(config['d']-1,0,-1):
    cores_new_2, R = rl_orthogonal_ind(cores, ranks, instructions, i)
    cores[i] = cores_new_2[i]
    cores[i-1] = cores_new_2[i-1]

In [17]:
cores_new[0].device

device(type='cpu')

In [18]:
cores_new[i - 1] = torch.zeros((1, 2, 3), device = cores_new[0].device)

In [13]:
torch.allclose(cores_new_2[i], cores_new[i]), torch.allclose(cores_new_2[i-1], cores_new[i-1])

(True, True)

In [12]:
# Check rl orthogonality
cores_new_left = cores_new[-2].transpose(0, 1)
cores_new_left.flatten(1) @ cores_new_left.flatten(1).T

tensor([[ 3.0000e+00, -1.1921e-07,  4.4703e-08,  1.4901e-08],
        [-1.1921e-07,  3.0000e+00, -5.2154e-08,  2.2352e-08],
        [ 4.4703e-08, -5.2154e-08,  3.0000e+00, -4.0978e-08],
        [ 1.4901e-08,  2.2352e-08, -4.0978e-08,  3.0000e+00]],
       grad_fn=<MmBackward0>)

In [28]:
cores_new[-2].transpose(0, 1).flatten(1)

tensor([[-0.0582, -0.0315,  0.2375,  ..., -0.0000, -0.0000, -0.0000],
        [-0.2869, -0.1762,  0.1234,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0928, -0.2206, -0.1132,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0517,  0.1162,  0.1581,  ...,  0.0000,  0.0000,  0.0000]],
       grad_fn=<UnsafeViewBackward0>)

In [61]:
trainer.model.get_submodule('model.model.func.etn.cores')[0] = cores_new[0]

In [63]:
cores_1 = trainer.model.get_submodule('model.model.func.etn.cores')[0]

cores_1.flatten(0, -2).T @ cores_1.flatten(0, -2)

tensor([[ 3.0000e+00, -1.5646e-07,  3.1665e-08,  1.1548e-07],
        [-1.5646e-07,  3.0000e+00,  1.1362e-07, -1.5274e-07],
        [ 3.1665e-08,  1.1362e-07,  3.0000e+00, -1.6764e-08],
        [ 1.1548e-07, -1.5274e-07, -1.6764e-08,  3.0000e+00]],
       grad_fn=<MmBackward0>)

In [ ]:
data_new['atomic_energy'].max()

In [7]:
trainer.train()

Number of weights: 4920
Number of trainable weights: 4920
instantiate Adam
        all_args :                                                 eps <-                               optimizer_params.eps
        all_args :                                        weight_decay <-                      optimizer_params.weight_decay
        all_args :                                               betas <-                             optimizer_params.betas
        all_args :                                             amsgrad <-                           optimizer_params.amsgrad
...Adam_param = dict(
...   optional_args = {'betas': (0.9, 0.999), 'eps': 1e-08, 'weight_decay': 0.0, 'amsgrad': False, 'foreach': None, 'maximize': False, 'capturable': False, 'differentiable': False, 'fused': None},
...   positional_args = {'params': <generator object Module.parameters at 0x7faddbc2a500>, 'lr': 0.001})
instantiate ReduceLROnPlateau
        all_args :                                              factor 

In [ ]:
  Train      #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! Train               2   52.071    0.001       0.0737       0.0235       0.0972         0.14        0.248         2.13       0.0815
! Validation          2   52.071    0.001       0.0601       0.0118       0.0719        0.137        0.238          1.4       0.0518
Wall time: 52.073412504047155
! Best model        2    0.072

In [48]:
list(trainer.model.model.model.children())

[SequentialGraphNetwork(
   (pair_type_embedding): PairTypeEmbedding()
   (one_hot): OneHotAtomEncoding()
   (radial_basis): RadialBasisEdgeEncoding(
     (basis): NormalizedBasis(
       (basis): BesselBasis()
     )
     (cutoff): PolynomialCutoff()
   )
   (spharm): SphericalHarmonicEdgeAttrs(
     (sh): SphericalHarmonics()
   )
   (edge_features_F): EdgeFeatures_F()
   (edge_f_sum): EdgewiseFSum()
   (etn): ETN_ALS_Module_opt(
     (cores): ParameterList(
         (0): Parameter containing: [torch.float32 of size 3x1x10x4]
         (1): Parameter containing: [torch.float32 of size 11x4x10x4]
         (2): Parameter containing: [torch.float32 of size 11x4x10x4]
         (3): Parameter containing: [torch.float32 of size 3x4x10x1]
     )
   )
   (per_species_rescale): PerSpeciesScaleShift()
   (total_energy_sum): AtomwiseReduce()
 )]

In [31]:
config['max_epochs']

2

In [17]:
config['N_rank_ett']

[4, 4, 4]

In [16]:

    

def rl_orthogonal(tt_cores, R, instr):
    """
    Orthogonalize the TT-cores right to left.

    Parameters
    ----------
    tt_cores : list of torch tensors.
        The TT-cores as a list.

    Returns
    -------
    tt_cores : list of torch tensors.
        The orthogonal TT-cores as a list.

    """  
    
    lmax = max([el[2] for el in instr])
    ind_left = [[i for i, el in enumerate(instr) if el[-1] == l] for l in range(lmax+1)]
    ind_right = [[i for i, el in enumerate(instr) if el[0] == l] for l in range(lmax+1)]
    
    d = len(tt_cores)

    rank_next = R[0]
    
    core_now = tt_cores[0]
    cores_new = d*[None]
        
    
    
    cores_new = d*[None]
    cores_new[-1] = tt_cores[-1]+0
    for i in range(d-1,0,-1):
        
        core_next = tt_cores[i - 1]
        for l in range(lmax+1):
            mode_shape = [cores_new[i].shape[2]]
            core_now = (torch.stack([cores_new[i][ind, ...] for ind in ind_right[l]], dim = -3).flatten(1)).t()
        
        
            Qmat, Rmat = QR(core_now)
            rnew = Rmat.shape[0]
            
            # update current core
            cores_new_tmp = tn.reshape(Qmat.T, [rnew]+[len(ind_right[l])] + mode_shape + [-1])
            cores_new[i][ind_right[l]] = cores_new_tmp.transpose(0, 1)
            
            R[i] = cores_new[i].shape[1]
            
            
            # and the i-1 one
            mode_shape = [core_next.shape[2]]
    
            core_next_tmp = tn.reshape(core_next[ind_left[l]],[len(ind_left[l])*core_next.shape[1]*core_next.shape[2],-1])
            core_next_tmp = core_next_tmp @ Rmat.T
            
            if l == 0:
                cores_new[i - 1] = tn.zeros([len(instr)] + [core_next.shape[1]] + mode_shape + [cores_new[i].shape[1]])
            
            cores_new[i - 1][ind_left[l]] = tn.reshape(core_next_tmp, [len(ind_left[l])] + [core_next.shape[1]] + mode_shape + [-1])
        
    return cores_new, R

In [8]:
trainer.model.get_submodule('model.model.func.etn.cores')

ParameterList(
    (0): Parameter containing: [torch.float32 of size 3x1x10x4]
    (1): Parameter containing: [torch.float32 of size 11x4x10x4]
    (2): Parameter containing: [torch.float32 of size 11x4x10x4]
    (3): Parameter containing: [torch.float32 of size 3x4x10x1]
)

In [37]:
trainer.model.get_buffer('model.model.func.etn.N_rank_ett')

tensor([4, 4, 4])

In [23]:
# Number of sweeps
nsw = 20
epochs_per_sweep = 20
assert (config['max_epochs'] % config['epochs_per_sweep'] == 0)
cur_sweep = 0

while cur_sweep < nsw:
    # Making all non trainable
    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(False)
    
    # Forward sweeps
    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(True)
        
        if i != 0:
            trainer.model.get_submodule('model.model.func.etn.cores')[i-1].requires_grad_(False)
        
        trainer.epoch_step()
        trainer.end_of_epoch_save()
        
        cur_sweep += 1
    
    # Backward sweeps   
    for i in range(config['d']-2, -1, -1):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(True)
        trainer.model.get_submodule('model.model.func.etn.cores')[i+1].requires_grad_(False)
        
        cur_sweep += 1
        trainer.epoch_step()
        trainer.end_of_epoch_save()

trainer.epoch_step()
trainer.end_of_epoch_save()


training
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae


tensor(0.9773)


      5   100        0.153        0.144      0.00928        0.191        0.326         1.35       0.0596
      5   200       0.0522       0.0266       0.0256       0.0819         0.14         2.79        0.065
      5   300      0.00917      0.00588      0.00329       0.0409       0.0658        0.488       0.0354
      5   400       0.0124      0.00998      0.00241       0.0558       0.0857        0.553       0.0344
      5   500       0.0376       0.0317      0.00595       0.0858        0.153         1.35        0.041

validation
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      5   100        0.042       0.0403      0.00168        0.111        0.172        0.546       0.0261


  Train      #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! Train               5  818.881    0.001       0.0465       0.0123       0.0589        0.108        0.195         1.5

tensor(0.9773)


      6   100      0.00563      0.00286      0.00277       0.0234       0.0459        0.517       0.0379
      6   200       0.0279       0.0173       0.0106       0.0603        0.113         1.79       0.0536
      6   300       0.0367       0.0286      0.00805          0.1        0.145         2.12       0.0562
      6   400        0.128        0.124      0.00437        0.144        0.302        0.666       0.0349
      6   500       0.0771       0.0729      0.00414        0.132        0.232         1.74       0.0487

validation
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      6   100       0.0417       0.0406      0.00112        0.108        0.173        0.502       0.0226


  Train      #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! Train               6  852.171    0.001       0.0431       0.0102       0.0532        0.103        0.188         1.4

...

     20   100      0.00668      0.00519      0.00149       0.0408       0.0618        0.752       0.0234
     20   200       0.0102      0.00277      0.00744       0.0262       0.0451         1.14       0.0626
     20   300       0.0338       0.0334     0.000421       0.0881        0.157         0.31        0.013
     20   400      0.00903      0.00513       0.0039       0.0396       0.0615         1.31       0.0499
     20   500      0.00533      0.00508     0.000255       0.0376       0.0612        0.242       0.0097

validation
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
     20   100       0.0247       0.0247     5.24e-05       0.0827        0.135        0.156      0.00504


  Train      #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! Train              20 1321.969    0.001       0.0283       0.0027        0.031        0.078        0.152         0.8

tensor(0.9672)


     21   100       0.0741       0.0705      0.00358        0.113        0.228         1.11       0.0457
     21   200      0.00564      0.00506     0.000573       0.0353       0.0611        0.258       0.0134
     21   300      0.00565       0.0035      0.00215       0.0281       0.0507        0.605       0.0319
     21   400       0.0374        0.034      0.00338        0.113        0.158         1.54       0.0375
     21   500      0.00708      0.00646     0.000614       0.0388        0.069        0.471       0.0184

validation
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
     21   100       0.0242       0.0241     9.84e-05       0.0822        0.133        0.238      0.00624


  Train      #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! Train              21 1356.103    0.001       0.0285      0.00225       0.0308       0.0782        0.152        0.73

tensor(0.9672)


     22   100       0.0184       0.0172      0.00123       0.0688        0.113         0.77       0.0258
     22   200      0.00786      0.00566      0.00221       0.0392       0.0646        0.821       0.0371
     22   300      0.00549       0.0054     9.08e-05       0.0421       0.0631         0.18      0.00737
     22   400         0.06       0.0591     0.000869        0.128        0.209        0.795        0.017
     22   500       0.0681       0.0659      0.00229        0.102         0.22        0.925       0.0275

validation
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
     22   100       0.0234       0.0233     8.63e-05       0.0822        0.131        0.228      0.00607


  Train      #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! Train              22 1390.044    0.001        0.027      0.00182       0.0289       0.0773         0.15        0.65

In [26]:
from nequip.utils import finish_all_writes

for callback in trainer._final_callbacks:
    callback(trainer)

trainer.final_log()

trainer.save()
finish_all_writes()

! Stop training: max epochs
Wall time: 2163.206009161
Cumulative wall time: 2163.206009161
Saved trainer to results/MEA_Allegro_2/example/trainer.pth
Saved last model to to results/MEA_Allegro_2/example/last_model.pth


In [11]:

trainer.epoch_step()
trainer.end_of_epoch_save()


training
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      5   100        0.153        0.145      0.00786        0.191        0.327          1.2       0.0547
      5   200       0.0489       0.0237       0.0252       0.0753        0.132         2.74       0.0597
      5   300      0.00897       0.0057      0.00327       0.0406       0.0648        0.458       0.0333
      5   400       0.0131       0.0101      0.00299       0.0559       0.0862        0.647       0.0419
      5   500       0.0356       0.0297      0.00592       0.0834        0.148         1.37        0.043

validation
# Epoch batch         loss       loss_f       loss_e        f_mae       f_rmse        e_mae      e/N_mae
      5   100       0.0423       0.0406      0.00165        0.111        0.173        0.544       0.0259


  Train      #    Epoch      wal       LR       loss_f       loss_e         loss        f_mae       f_rmse        e_mae      e/N_mae
! T

In [10]:
def batch_step(data, validation=False):
    # no need to have gradients from old steps taking up memory
    #self.optim.zero_grad(set_to_none=True)

    #if validation:
    #    self.model.eval()
    #else:
    #    self.model.train()

    # Do any target rescaling
    data = AtomicData.to_AtomicDataDict(data)

    # this will normalize the targets
    # in both validation and train we want targets normalized _for the loss_
    data_for_loss = trainer.model.unscale(data, force_process=True)

    # Run model
    # We make a shallow copy of the input dict in case the model modifies it
    out = trainer.model(data_for_loss)
    #print(out)
    return out

In [12]:
from nequip.train._key import ABBREV, LOSS_KEY, TRAIN, VALIDATION

def epoch_step(trainer):

    dataloaders = {TRAIN: trainer.dl_train, VALIDATION: trainer.dl_val}
    categories = [TRAIN, VALIDATION] if trainer.iepoch >= 0 else [VALIDATION]
    dataloaders = [
        dataloaders[c] for c in categories
    ]  # get the right dataloaders for the catagories we actually run
    if TRAIN in categories:
        # We have to step the sampler so it knows what epoch it is
        trainer.dl_train_sampler.step_epoch(trainer.iepoch)

    #self.metrics_dict = {}
    #self.loss_dict = {}

    for category, dataset in zip(categories, dataloaders):
        
        for trainer.ibatch, batch in enumerate(dataset):
            print(trainer.ibatch, batch)
            out = batch_step(
                data=batch,
                validation=(category == VALIDATION),
            )
            
    return out, batch

In [13]:
len(dataset[:20])

20

In [14]:
dataset[:20]

ASEDataset(20)

In [15]:
trainer.n_train

5000

In [16]:
trainer.dl_val.batch_size

5

In [19]:
trainer.n_train = 2
trainer.n_val = 3

trainer.train_idcs = torch.tensor([201, 1201], dtype = torch.long)
trainer.val_idcs = torch.tensor([301, 1401, 5], dtype = torch.long)

trainer.set_dataset(dataset, None)

In [20]:
out, batch = epoch_step(trainer)

0 Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])


In [21]:
out['pos'].shape

torch.Size([34, 3])

In [22]:
out['batch']

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 2, 2])

In [23]:
out['ptr']

tensor([ 0, 16, 32, 34])

In [24]:
print(AtomicData.to_AtomicDataDict(dataset[301])['pos'].shape)
print(AtomicData.to_AtomicDataDict(dataset[1401])['pos'].shape)
print(AtomicData.to_AtomicDataDict(dataset[5])['pos'].shape)

torch.Size([16, 3])
torch.Size([16, 3])
torch.Size([2, 3])


In [25]:
out.keys()

dict_keys(['edge_index', 'pos', 'batch', 'ptr', 'cell', 'edge_cell_shift', 'atom_types', 'edge_vectors', 'edge_types', 'node_attrs', 'node_features', 'edge_lengths', 'edge_embedding', 'edge_cutoff', 'edge_attrs', 'edge_features_F', 'node_features_F', 'node_features_ETN', 'atomic_energy', 'total_energy', 'forces', 'stress', 'virial', 'atom_virial'])

In [28]:
trainer.batch_metrics = trainer.metrics(pred=out, ref=batch)

ValueError: Data shape of batch, torch.Size([3, 34]), does not match the input data dimension of this RunningStats, torch.Size([192])

In [29]:
batch

Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])

In [30]:
for key in out:
    print(key, out[key].shape)

edge_index torch.Size([2, 884])
pos torch.Size([34, 3])
batch torch.Size([34])
ptr torch.Size([4])
cell torch.Size([3, 3, 3])
edge_cell_shift torch.Size([884, 3])
atom_types torch.Size([34, 1])
edge_vectors torch.Size([884, 3])
edge_types torch.Size([884, 1])
node_attrs torch.Size([34, 4])
node_features torch.Size([34, 4])
edge_lengths torch.Size([884])
edge_embedding torch.Size([884, 8])
edge_cutoff torch.Size([884, 1])
edge_attrs torch.Size([884, 9])
edge_features_F torch.Size([884, 9, 10])
node_features_F torch.Size([34, 9, 10])
node_features_ETN torch.Size([34, 9, 10])
atomic_energy torch.Size([34, 34])
total_energy torch.Size([3, 34])
forces torch.Size([34, 3])
stress torch.Size([3, 3, 3])
virial torch.Size([3, 3, 3])
atom_virial torch.Size([34, 3, 3])


In [31]:
batch

Batch(atom_types=[34, 1], batch=[34], cell=[3, 3, 3], edge_cell_shift=[884, 3], edge_index=[2, 884], forces=[34, 3], pbc=[3, 3], pos=[34, 3], ptr=[4], stress=[3, 3, 3], total_energy=[3, 1])

In [32]:
out['total_energy'].shape

torch.Size([3, 34])

In [40]:

( data_new[_keys.NODE_FEATURES_ETN] * data_new[_keys.NODE_FEATURES_ETN] ).sum(dim = (-2, -1)).unsqueeze(-1)

tensor([[5.7274e-15],
        [5.7274e-15]], grad_fn=<UnsqueezeBackward0>)

In [37]:
data_new["node_features_F"].shape

torch.Size([2, 9, 10])